# Study 0 — Robustness Checks

Tuned baselines (Optuna), multi-seed variance, and walk-forward validation.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
import os

# Retrieve the GitHub token from Colab Secrets
github_token = userdata.get('GITHUB_PAT')

username = "elroy-galbraith"
repository = "fin-jepa"
repo_url = f"https://{github_token}@github.com/{username}/{repository}.git"

if not os.path.exists(f'/content/{repository}'):
    !git clone {repo_url} /content/{repository}

%cd /content/{repository}

# Install fin-jepa as editable WITHOUT upgrading Colab's pre-installed deps
!pip install -q --no-deps -e '.[dev]'

# Install only the deps Colab doesn't already ship
!pip install -q hydra-core omegaconf mlflow optuna xgboost yfinance rich python-dotenv tqdm

In [ ]:
# Symlink data from Google Drive so relative paths work
import os

DRIVE_DATA = '/content/drive/MyDrive/Fin-JEPA/data'

for subdir in ['raw', 'processed']:
    target = f'data/{subdir}'
    if os.path.islink(target) or os.path.isdir(target):
        !rm -rf {target}
    os.symlink(f'{DRIVE_DATA}/{subdir}', target)

# Verify data exists
!ls -la data/raw/xbrl_features.parquet
!ls -la data/processed/label_database.parquet
!ls -la data/raw/market/market_aligned.parquet
print('Data linked successfully!')
# Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU detected! Change runtime to T4 or A100.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_memory / 1e9:.1f} GB')

In [ ]:
# Common setup: logging, configs, control flag
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import OmegaConf

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# Set to True to re-run experiments even if cached results exist
FORCE_RERUN = False

# Load configs from YAML (override individual values below if needed)
benchmark_cfg = OmegaConf.load('configs/study0/benchmark.yaml')
ssl_cfg = OmegaConf.load('configs/study0/ssl_experiment.yaml')

# Paths
RESULTS_DIR = Path('results/study0')
BENCHMARK_PATH = RESULTS_DIR / 'benchmark_results.json'
SWEEP_PATH = RESULTS_DIR / 'ft_sweep_results.json'
SSL_V2_PATH = RESULTS_DIR / 'ssl_experiment_results_v2.json'
FINAL_PATH = RESULTS_DIR / 'final_benchmark.json'

OUTCOMES = ['stock_decline', 'earnings_restate', 'audit_qualification',
            'sec_enforcement', 'bankruptcy']

print(f'Results dir: {RESULTS_DIR}')
print(f'Benchmark cached: {BENCHMARK_PATH.exists()}')
print(f'Sweep cached: {SWEEP_PATH.exists()}')
print(f'SSL v2 cached: {SSL_V2_PATH.exists()}')
print(f'FORCE_RERUN: {FORCE_RERUN}')
ARTIFACT_DIR = Path('/content/drive/MyDrive/Fin-JEPA/artifacts/study0')
ARTIFACT_PATH = ARTIFACT_DIR / 'preprocessing_v1.pkl'

In [ ]:
import pickle

if ARTIFACT_PATH.exists():
    with open(ARTIFACT_PATH, 'rb') as f:
        artifact = pickle.load(f)
    print(f'Loaded preprocessing artifact from {ARTIFACT_PATH}')
    print(f'  Keys: {list(artifact.keys())}')
else:
    print(f'No preprocessing artifact at {ARTIFACT_PATH} — will build from scratch')
    artifact = None

In [ ]:
# Per-section rerun flags (override individual sections without re-running all)
FORCE_RERUN_TUNED_BASELINES = FORCE_RERUN
FORCE_RERUN_TUNED_FTT = FORCE_RERUN
FORCE_RERUN_MULTISEED = FORCE_RERUN
FORCE_RERUN_MULTISEED_SSL = FORCE_RERUN
FORCE_RERUN_WALKFORWARD = FORCE_RERUN

# Load prior results needed by this notebook
SWEEP_PATH = RESULTS_DIR / 'ft_sweep_results.json'
SSL_V2_PATH = RESULTS_DIR / 'ssl_experiment_results_v2.json'
FINAL_RESULT_PATH = RESULTS_DIR / 'final_benchmark.json'

for path, nb in [(SWEEP_PATH, '02'), (SSL_V2_PATH, '03')]:
    if not path.exists():
        raise FileNotFoundError(f'{path.name} not found. Run {nb}_*.ipynb first.')

with open(SWEEP_PATH) as f:
    sweep_results = json.load(f)
best_cfg = sweep_results['best_config']

with open(SSL_V2_PATH) as f:
    ssl_results = json.load(f)

# SSL checkpoint for multi-seed
ssl_ckpt_dir = Path('models/checkpoints/study0_ssl_experiment_v2')

print(f"Loaded sweep (best: d_token={best_cfg['d_token']}, n_layers={best_cfg['n_layers']})")
print(f"Loaded SSL v2 results")

## 7. Tuned Baselines (Optuna), Multi-Seed & Walk-Forward

### 7a. Optuna-Tuned Baselines

Re-run the baseline benchmark with Optuna temporal-CV tuning enabled (30 trials, 3-fold expanding-window CV). Computational budget matches the FT-Transformer grid search (~30 configs per model).

In [ ]:
%%time
from fin_jepa.training.train_study0 import run_benchmark

# Override config to enable Optuna tuning
tuned_cfg = OmegaConf.to_container(benchmark_cfg, resolve=True)
tuned_cfg['baselines'] = dict(tuned_cfg.get('baselines', {}))
tuned_cfg['baselines']['tune'] = True
tuned_cfg['baselines']['n_trials'] = 30
tuned_cfg['baselines']['n_cv_splits'] = 3
tuned_cfg['results_dir'] = str(RESULTS_DIR)

tuned_baseline_results = run_benchmark(tuned_cfg)

print('Gate:', 'PASSED' if tuned_baseline_results['gate']['passed'] else 'FAILED',
      f'({tuned_baseline_results["gate"]["n_wins"]}/5 wins)')


In [ ]:
# Compare tuned baselines vs. sweep-winning FT-Transformer
import pandas as pd

best = sweep_results['best_config']
rows = []
for oc, res in tuned_baseline_results['outcomes'].items():
    xgb_tuned = res.get('xgboost', {}).get('auroc', float('nan'))
    gbt_tuned = res.get('gbt_raw', {}).get('auroc', float('nan'))
    lr_tuned = res.get('lr_full', {}).get('auroc', float('nan'))
    # Sweep-winning FT from section 3
    ft_sweep = None
    for cfg in sweep_results.get('configs', []):
        if (cfg.get('d_token') == best['d_token'] and
            cfg.get('n_layers') == best['n_layers'] and
            cfg.get('lr') == best['lr']):
            ft_sweep = cfg.get('outcomes', {}).get(oc, {}).get('auroc')
            break
    # SSL FT from section 5 final benchmark
    ft_ssl = None
    try:
        ft_ssl = final_results.get('ft_ssl', {}).get(oc, {}).get('auroc')
    except NameError:
        pass
    rows.append({
        'outcome': oc,
        'XGB (tuned)': round(xgb_tuned, 4) if xgb_tuned == xgb_tuned else None,
        'GBT (tuned)': round(gbt_tuned, 4) if gbt_tuned == gbt_tuned else None,
        'LR (tuned)': round(lr_tuned, 4) if lr_tuned == lr_tuned else None,
        'FT sweep': round(ft_sweep, 4) if ft_sweep else None,
        'FT+SSL': round(ft_ssl, 4) if ft_ssl else None,
    })
tuned_df = pd.DataFrame(rows).set_index('outcome')
display(tuned_df.style.highlight_max(axis=1, color='#c6efce'))

# The real gate comparison: tuned XGB vs sweep/SSL FT-Transformer
print('\n=== Tuned XGB vs. Best FT-Transformer (sweep + SSL) ===')
for idx_row, row in tuned_df.iterrows():
    xgb_v = row.get('XGB (tuned)')
    ft_v = row.get('FT+SSL') or row.get('FT sweep')
    if xgb_v and ft_v:
        delta = ft_v - xgb_v
        tag = 'WIN' if delta >= 0.01 else ('tie' if abs(delta) < 0.01 else 'LOSS')
        print(f'  {idx_row}: XGB={xgb_v:.4f}  FT={ft_v:.4f}  delta={delta:+.4f}  [{tag}]')


In [ ]:
# --- Figure: Tuned-baseline hero chart ---
import matplotlib.pyplot as plt
import pandas as pd

tuned_models = ['LR (trad)', 'LR (full)', 'XGB (tuned)', 'GBT (tuned)', 'FT-Transformer']
tuned_rows = []
for oc in OUTCOMES:
    oc_res = tuned_baseline_results['outcomes'].get(oc, {})
    row = {'Outcome': oc}
    row['LR (trad)'] = oc_res.get('lr_traditional', {}).get('auroc')
    row['LR (full)'] = oc_res.get('lr_full', {}).get('auroc')
    row['XGB (tuned)'] = oc_res.get('xgboost', {}).get('auroc')
    row['GBT (tuned)'] = oc_res.get('gbt_raw', {}).get('auroc')
    row['FT-Transformer'] = oc_res.get('ft_transformer', {}).get('auroc')
    tuned_rows.append(row)

tuned_plot_df = pd.DataFrame(tuned_rows).set_index('Outcome')
tuned_plot_df = tuned_plot_df[[c for c in tuned_models if c in tuned_plot_df.columns]].dropna(how='all')

if not tuned_plot_df.empty:
    colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f']
    ax = tuned_plot_df.plot(kind='bar', figsize=(14, 6), width=0.85,
                            color=colors[:len(tuned_plot_df.columns)], edgecolor='white')
    ax.set_ylabel('Test AUROC')
    ax.set_title('Study 0: All Models with Optuna-Tuned Baselines (Test Set)')
    ax.set_ylim(0.35, max(tuned_plot_df.max().max() + 0.05, 0.8))
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.legend(loc='upper right', fontsize=9)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    fig_dir = RESULTS_DIR / 'figures'
    fig_dir.mkdir(exist_ok=True)
    plt.savefig(fig_dir / 'tuned_baseline_auroc.png', bbox_inches='tight', dpi=300)
    plt.show()
    print('Saved tuned_baseline_auroc.png')
else:
    print('No tuned baseline results to plot.')

### 7a½. Optuna-Tuned FT-Transformer (ATS-251)

Tune FT-Transformer hyperparameters (`learning_rate`, `n_layers`, `d_token`) using Optuna with temporal CV, mirroring the baseline tuning in 7a. The tuned params feed into walk-forward evaluation (section 7c).

> **Runtime:** ~20 min on T4 (30 trials × 3 CV folds × 50 epochs each).

In [ ]:
%%time
from fin_jepa.training.train_study0 import tune_ft_transformer

# Use stock_decline for tuning — it has the most positives and is
# representative; architecture choices (d_token, n_layers) are
# outcome-agnostic for the shared encoder.
tune_outcome = 'stock_decline'
y_tune = splits['train'][tune_outcome].values.astype(float)

ft_search = OmegaConf.to_container(benchmark_cfg, resolve=True).get(
    'ft_transformer_search', {
        'learning_rate': {'type': 'float', 'low': 3e-5, 'high': 1e-3, 'log': True},
        'n_layers': {'type': 'categorical', 'choices': [2, 3, 4]},
        'd_token': {'type': 'categorical', 'choices': [128, 192, 256]},
    }
)

FT_TUNE_PATH = RESULTS_DIR / 'ft_transformer_tuning.json'
if not FORCE_RERUN_TUNED_FTT and FT_TUNE_PATH.exists():
    print(f'Loading cached FT tuning results from {FT_TUNE_PATH}')
    with open(FT_TUNE_PATH) as f:
        tuned_ft_result = json.load(f)
else:
    tuned_ft_result = tune_ft_transformer(
        X_train_df=splits['train'],
        y_train=y_tune,
        feature_cols=feature_cols,
        device=device,
        search_space=ft_search,
        n_splits=3,
        n_trials=30,
        seed=SEED,
        n_cat=n_cat,
        cat_cards=cat_cards,
        categorical_cols=categorical_cols,
    )
    # Cache results
    FT_TUNE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(FT_TUNE_PATH, 'w') as f:
        json.dump(tuned_ft_result, f, indent=2, default=str)
    print(f'Saved FT tuning results to {FT_TUNE_PATH}')

tuned_ft_params = tuned_ft_result['best_params']
print(f'\nBest FT-Transformer params: {tuned_ft_params}')
print(f'Best mean val AUROC: {tuned_ft_result["mean_val_auroc"]:.4f}')

# Compare with grid-search winner from section 3
best_grid = sweep_results['best_config']
print(f'\nGrid-search winner:  lr={best_grid["lr"]}, '
      f'n_layers={best_grid["n_layers"]}, d_token={best_grid["d_token"]}')
print(f'Optuna winner:       lr={tuned_ft_params.get("learning_rate", "n/a")}, '
      f'n_layers={tuned_ft_params.get("n_layers", "n/a")}, '
      f'd_token={tuned_ft_params.get("d_token", "n/a")}')

### 7b. Multi-Seed FT-Transformer Variance

Train FT-Transformer with 3 different random seeds and report mean ± std AUROC per outcome.
Runs both **scratch** and **best SSL** variants to quantify variance from random initialisation.


In [ ]:
%%time
from fin_jepa.training.train_study0 import run_multiseed_benchmark

multiseed_cfg = OmegaConf.to_container(benchmark_cfg, resolve=True)
multiseed_cfg['results_dir'] = str(RESULTS_DIR)
SEEDS = [42, 123, 456]

# ATS-217: inject sweep-winning architecture + lr so the multiseed
# FT-Transformer matches Cell 24's final_benchmark ft_scratch exactly.
best = sweep_results['best_config']
multiseed_cfg.setdefault('ft_transformer', {}).update({
    'd_token': best['d_token'],
    'n_heads': best['n_heads'],
    'n_layers': best['n_layers'],
})
multiseed_cfg.setdefault('training', {})['learning_rate'] = best['lr']

# --- Scratch variant ---
# ATS-217: pass the shared preprocessing pipeline so seed-42
# numbers match the main benchmark (Tables 3/5).
multiseed_results = run_multiseed_benchmark(
    multiseed_cfg, seeds=SEEDS,
    prebuilt_splits=splits,
    prebuilt_feature_cols=feature_cols,
    prebuilt_cat_cols=categorical_cols,
)
print(f'Scratch seeds: {multiseed_results["seeds"]}')

# --- SSL-pretrained variant ---
ssl_ckpt_dir = Path('models/checkpoints/study0_ssl_experiment_v2')
best_mr_key = None
if 'ssl_results' in dir() and 'pretrained' in ssl_results:
    mask_ratios_ms = sorted(ssl_results.get('pretrained', {}).keys())
    best_mr_auroc = -1
    for mr in mask_ratios_ms:
        aurocs = [ssl_results['pretrained'].get(mr, {}).get(o, {}).get('auroc', 0)
                  for o in OUTCOMES]
        avg = np.mean([a for a in aurocs if a and a > 0]) if any(a and a > 0 for a in aurocs) else 0
        if avg > best_mr_auroc:
            best_mr_auroc = avg
            best_mr_key = mr

ssl_ckpt_path = ssl_ckpt_dir / f'encoder_mr{best_mr_key}.pt' if best_mr_key else None

multiseed_ssl_results = None
if ssl_ckpt_path and ssl_ckpt_path.exists():
    ssl_ms_cfg = dict(multiseed_cfg)  # inherits sweep arch + lr
    ssl_ms_cfg['ssl_checkpoint'] = str(ssl_ckpt_path)
    # ATS-217: SSL variant also uses shared splits
    multiseed_ssl_results = run_multiseed_benchmark(
        ssl_ms_cfg, seeds=SEEDS,
        prebuilt_splits=splits,
        prebuilt_feature_cols=feature_cols,
        prebuilt_cat_cols=categorical_cols,
    )
    print(f'SSL seeds: {multiseed_ssl_results["seeds"]}')
else:
    print('No SSL checkpoint found -- skipping SSL multi-seed variant')


In [ ]:
# Display mean +/- std AUROC per outcome (scratch + SSL)
import pandas as pd

rows = []
for oc, stats in multiseed_results['multiseed'].items():
    row = {
        'outcome': oc,
        'scratch mean': round(stats['mean_auroc'], 4),
        'scratch std': round(stats['std_auroc'], 4),
        'scratch 95% CI': f'{stats["mean_auroc"]:.4f} +/- {2*stats["std_auroc"]:.4f}',
    }
    if multiseed_ssl_results:
        ssl_stats = multiseed_ssl_results['multiseed'].get(oc, {})
        if ssl_stats:
            row['SSL mean'] = round(ssl_stats['mean_auroc'], 4)
            row['SSL std'] = round(ssl_stats['std_auroc'], 4)
            row['SSL 95% CI'] = f'{ssl_stats["mean_auroc"]:.4f} +/- {2*ssl_stats["std_auroc"]:.4f}'
    rows.append(row)
pd.DataFrame(rows).set_index('outcome')


In [ ]:
# --- Figure: Multi-seed variance ---
import matplotlib.pyplot as plt
import numpy as np

outcomes_ms = list(multiseed_results['multiseed'].keys())
means = [multiseed_results['multiseed'][oc]['mean_auroc'] for oc in outcomes_ms]
stds = [multiseed_results['multiseed'][oc]['std_auroc'] for oc in outcomes_ms]
per_seed = {s: [multiseed_results['multiseed'][oc]['per_seed_auroc'][str(s)]
                for oc in outcomes_ms] for s in multiseed_results['seeds']}

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(outcomes_ms))
width = 0.15
markers = ['o', 's', 'D']
seed_colors = ['#4e79a7', '#f28e2b', '#e15759']

# Plot individual seeds
for j, (seed, vals) in enumerate(per_seed.items()):
    ax.scatter(x + (j - 1) * width, vals, marker=markers[j], color=seed_colors[j],
               s=60, zorder=3, label=f'Seed {seed}')

# Plot mean +/- std as error bars
ax.errorbar(x, means, yerr=[2*s for s in stds], fmt='_', color='black',
            capsize=6, capthick=2, linewidth=2, zorder=4, label='Mean +/- 2*std')

ax.set_xticks(x)
ax.set_xticklabels(outcomes_ms, rotation=30, ha='right')
ax.set_ylabel('Test AUROC')
ax.set_title('FT-Transformer Multi-Seed Variance (3 Seeds, From Scratch)')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()

fig_dir = RESULTS_DIR / 'figures'
fig_dir.mkdir(exist_ok=True)
plt.savefig(fig_dir / 'multiseed_variance.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved multiseed_variance.png')

### 7b½. Multi-Seed SSL Evaluation (ATS-252)

Run SSL pretraining + fine-tuning across 3 seeds × 3 mask ratios. Reports mean ± std AUROC per (outcome, mask_ratio) to confirm which SSL gains survive initialisation variance.

> **Runtime:** ~45 min on T4 (3 seeds × 3 mask ratios × 200 pretrain epochs + 5 outcomes × fine-tune).

In [ ]:
%%time
from fin_jepa.training.pretrain_ssl import run_multiseed_ssl

ms_ssl_cfg = OmegaConf.to_container(ssl_cfg, resolve=True)
ms_ssl_cfg['results_dir'] = str(RESULTS_DIR)

# Inject sweep-winning architecture so SSL uses the same arch as scratch
best = sweep_results['best_config']
ms_ssl_cfg.setdefault('ft_transformer', {}).update({
    'd_token': best['d_token'],
    'n_heads': best['n_heads'],
    'n_layers': best['n_layers'],
})

MULTISEED_SSL_PATH = RESULTS_DIR / 'multiseed_ssl.json'
if not FORCE_RERUN_MULTISEED_SSL and MULTISEED_SSL_PATH.exists():
    print(f'Loading cached multi-seed SSL results from {MULTISEED_SSL_PATH}')
    with open(MULTISEED_SSL_PATH) as f:
        multiseed_ssl = json.load(f)
else:
    multiseed_ssl = run_multiseed_ssl(
        ms_ssl_cfg,
        seeds=[42, 123, 456],
        prebuilt_splits=splits,
        prebuilt_feature_cols=feature_cols,
        prebuilt_cat_cols=categorical_cols,
    )
    print(f'Multi-seed SSL complete: {len(multiseed_ssl["seeds"])} seeds '
          f'x {len(multiseed_ssl["mask_ratios"])} mask ratios')

In [ ]:
# Multi-seed SSL results table: mean +/- std AUROC per (outcome, variant)
import pandas as pd

rows = []
for oc, variants in multiseed_ssl['multiseed_ssl'].items():
    row = {'outcome': oc}
    for key in ['scratch', '0.15', '0.30', '0.50']:
        stats = variants.get(key, {})
        if stats:
            label = 'scratch' if key == 'scratch' else f'mr={key}'
            row[f'{label} mean'] = round(stats['mean_auroc'], 4)
            row[f'{label} std'] = round(stats['std_auroc'], 4)
    rows.append(row)

ms_ssl_df = pd.DataFrame(rows).set_index('outcome')
mean_cols = [c for c in ms_ssl_df.columns if 'mean' in c]
display(ms_ssl_df.style.highlight_max(axis=1, subset=mean_cols, color='#c6efce'))

# Best mask ratio per outcome
print('\n=== Best variant per outcome ===')
for oc, variants in multiseed_ssl['multiseed_ssl'].items():
    if not variants:
        print(f'  {oc}: skipped (no variants)')
        continue
    best_k, best_v = max(variants.items(), key=lambda kv: kv[1].get('mean_auroc', 0))
    label = 'scratch' if best_k == 'scratch' else f'SSL mr={best_k}'
    print(f'  {oc}: {label} (mean={best_v["mean_auroc"]:.4f} +/- {best_v["std_auroc"]:.4f})')

In [ ]:
# --- Figure: Multi-seed SSL AUROC by mask ratio ---
import matplotlib.pyplot as plt
import numpy as np

outcomes_ssl = list(multiseed_ssl['multiseed_ssl'].keys())
variants = ['scratch', '0.15', '0.30', '0.50']
variants = [v for v in variants
            if any(v in multiseed_ssl['multiseed_ssl'][oc] for oc in outcomes_ssl)]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(outcomes_ssl))
width = 0.18
colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']

for j, var in enumerate(variants):
    means = [multiseed_ssl['multiseed_ssl'][oc].get(var, {}).get('mean_auroc', 0)
             for oc in outcomes_ssl]
    stds = [multiseed_ssl['multiseed_ssl'][oc].get(var, {}).get('std_auroc', 0)
            for oc in outcomes_ssl]
    label = 'scratch' if var == 'scratch' else f'SSL mr={var}'
    ax.bar(x + j * width, means, width, yerr=[2*s for s in stds],
           label=label, color=colors[j % len(colors)], capsize=4, edgecolor='white')

ax.set_xticks(x + width * (len(variants) - 1) / 2)
ax.set_xticklabels(outcomes_ssl, rotation=30, ha='right')
ax.set_ylabel('Test AUROC')
ax.set_title('Multi-Seed SSL: Mean +/- 2\u03c3 by Mask Ratio (3 Seeds)')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()

fig_dir = RESULTS_DIR / 'figures'
fig_dir.mkdir(exist_ok=True)
plt.savefig(fig_dir / 'multiseed_ssl_auroc.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved multiseed_ssl_auroc.png')

### 7c. Walk-Forward (Expanding-Window) Validation

Expanding-window evaluation across multiple train/test cutoffs (2014–2023). Each fold expands the training set by 1 year and tests on the next 2-year window. Reports XGBoost and FT-Transformer AUROC per fold per outcome.

In [ ]:
%%time
from fin_jepa.training.train_study0 import run_walk_forward

wf_cfg = OmegaConf.to_container(benchmark_cfg, resolve=True)
wf_cfg['results_dir'] = str(RESULTS_DIR)

# Pass Optuna-tuned FT-Transformer params from section 7a½ (ATS-253).
# XGBoost uses config defaults — the tuned baseline run in 7a already
# validated that tuning doesn't change the gate outcome.
_tuned_ft = tuned_ft_params if 'tuned_ft_params' in dir() else None

try:
    wf_results = run_walk_forward(wf_cfg, tuned_ft_params=_tuned_ft)
    print(f'Walk-forward: {wf_results["n_folds"]} folds completed')
    if _tuned_ft:
        print(f'FT-Transformer used tuned params: {_tuned_ft}')
    else:
        print('FT-Transformer used config defaults (no tuned params available)')
except ValueError as e:
    print(f"Walk-forward failed due to an empty data split: {e}")
    print("Skipping walk-forward validation.")
    wf_results = {"walk_forward_folds": [], "n_folds": 0}

In [ ]:
# Display walk-forward AUROC table: rows = folds, cols = outcome × model
import pandas as pd
import numpy as np

rows = []
for fold in wf_results["walk_forward_folds"]:
    row = {"fold": fold["label"]}
    for oc, res in fold["outcomes"].items():
        row[f"{oc}|XGB"] = round(res["xgb_auroc"], 4)
        row[f"{oc}|FT"] = round(res["ft_auroc"], 4)
    rows.append(row)

if not rows:
    print("No walk-forward folds produced results — check data coverage.")
else:
    wf_df = pd.DataFrame(rows).set_index("fold")
    display(wf_df)

    ft_cols = [c for c in wf_df.columns if c.endswith("|FT")]
    xgb_cols = [c for c in wf_df.columns if c.endswith("|XGB")]
    print(f"Mean FT AUROC across folds:  {wf_df[ft_cols].mean().mean():.4f}")
    print(f"Mean XGB AUROC across folds: {wf_df[xgb_cols].mean().mean():.4f}")
